# QMS: Google Sheet → BigQuery (native table)

Runs in **Google Colab** as **you** (not the Vercel service account).

- Reads the Task Tracker tab from Google Sheets
- Loads into **native** BigQuery table with **WRITE_TRUNCATE** (full replace)
- Uses all-string columns so free-text `qcComment` does not break CSV

**Target:** `gen-lang-client-0732074273.qms_dashboard.task_tracker` @ **US**

### Before you run
1. You can open the Google Sheet in the browser
2. Your Google account has **BigQuery Data Editor** + **Job User** on the GCP project
3. Dataset `qms_dashboard` exists in location **US**

### Not real-time
This is a snapshot. Re-run this notebook (or schedule it) whenever you need fresh data.

## 1) Config — edit these

In [ ]:
# === EDIT THESE ===
PROJECT_ID = "gen-lang-client-0732074273"
DATASET_ID = "qms_dashboard"
TABLE_ID = "task_tracker"
LOCATION = "US"  # must match dataset location

# From the Sheet URL: https://docs.google.com/spreadsheets/d/SPREADSHEET_ID/edit
SPREADSHEET_ID = "PASTE_YOUR_SPREADSHEET_ID_HERE"

# Exact tab name at the bottom of the sheet
SHEET_NAME = "Task Tracker"

print("Config OK")
print(f"Destination: {PROJECT_ID}.{DATASET_ID}.{TABLE_ID} @ {LOCATION}")

## 2) Install packages + authenticate

When prompted, sign in with the **same Google account** that can open the Sheet and write BigQuery.

In [ ]:
!pip -q install google-cloud-bigquery google-cloud-bigquery-storage gspread google-auth pandas pyarrow

from google.colab import auth
auth.authenticate_user()
print("Authenticated as Colab user")

## 3) Read Google Sheet

In [ ]:
import re
import time
from datetime import datetime, timezone

import gspread
import pandas as pd
from google.auth import default
from google.cloud import bigquery

started = datetime.now(timezone.utc)
print(f"[{started.isoformat()}] START sync")

creds, _ = default(scopes=[
    "https://www.googleapis.com/auth/spreadsheets.readonly",
    "https://www.googleapis.com/auth/drive.readonly",
    "https://www.googleapis.com/auth/bigquery",
    "https://www.googleapis.com/auth/cloud-platform",
])

if not SPREADSHEET_ID or SPREADSHEET_ID.startswith("PASTE_"):
    raise ValueError("Set SPREADSHEET_ID from your Sheet URL (the long id between /d/ and /edit).")

gc = gspread.authorize(creds)
print(f"[{datetime.now(timezone.utc).isoformat()}] Opening spreadsheet…")
sh = gc.open_by_key(SPREADSHEET_ID)

try:
    ws = sh.worksheet(SHEET_NAME)
except gspread.WorksheetNotFound as e:
    tabs = [w.title for w in sh.worksheets()]
    raise RuntimeError(
        f'Tab "{SHEET_NAME}" not found. Available tabs: {tabs}. '
        "Update SHEET_NAME and re-run."
    ) from e

print(f"[{datetime.now(timezone.utc).isoformat()}] Reading tab '{SHEET_NAME}' (this can take 1–3 min)…")
t0 = time.time()
# get_all_values = display strings (good for mixed dates/comments)
rows = ws.get_all_values()
print(f"[{datetime.now(timezone.utc).isoformat()}] Read done in {time.time()-t0:.1f}s — rows={len(rows)}")

if len(rows) < 2:
    raise RuntimeError("Sheet has no data rows (need header + data).")

header = rows[0]
data = rows[1:]
df = pd.DataFrame(data, columns=header)

# Drop fully empty rows
df = df.replace("", pd.NA).dropna(how="all").fillna("")
print(f"Data rows after drop-empty: {len(df)} | cols: {len(df.columns)}")
print("Sample columns:", list(df.columns)[:12], "…")
df.head(2)

## 4) Clean column names for BigQuery

In [ ]:
def sanitize_field_name(name: str, index: int) -> str:
    s = str(name or "").replace("\ufeff", "").strip()
    s = re.sub(r"[^A-Za-z0-9_]", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    if not s:
        s = f"col_{index}"
    if re.match(r"^[0-9]", s):
        s = f"c_{s}"
    return s[:300]

used = {}
new_cols = []
for i, col in enumerate(df.columns):
    base = sanitize_field_name(col, i)
    name = base
    n = 2
    while name in used:
        name = f"{base}_{n}"
        n += 1
    used[name] = True
    new_cols.append(name)

df.columns = new_cols

# All string — safest for free-text comments / mixed types
df = df.astype(str)
# Clean zero-width junk often pasted into QC comments
for c in df.columns:
    df[c] = df[c].str.replace(r"[\u200b-\u200d\ufeff\u2060]", "", regex=True)

print("Sanitized columns:", list(df.columns)[:15], "…")
print(f"Final shape: {df.shape}")

## 5) Load into BigQuery (full replace)

In [ ]:
table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"
client = bigquery.Client(project=PROJECT_ID, credentials=creds, location=LOCATION)

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    # All STRING schema from dataframe dtypes
    autodetect=False,
    schema=[bigquery.SchemaField(name, "STRING") for name in df.columns],
)

print(f"[{datetime.now(timezone.utc).isoformat()}] Loading {len(df):,} rows → {table_id} (WRITE_TRUNCATE)…")
t0 = time.time()
job = client.load_table_from_dataframe(df, table_id, job_config=job_config, location=LOCATION)
print(f"Job started: {job.job_id}")

result = job.result()  # waits until done
elapsed = time.time() - t0
print(f"[{datetime.now(timezone.utc).isoformat()}] Load DONE in {elapsed:.1f}s  state={job.state}")

if job.errors:
    print("Job errors:", job.errors)
    raise RuntimeError(f"BigQuery load completed with errors: {job.errors}")

table = client.get_table(table_id)
print(f"Table type/location: {table.table_type} / {table.location}")
print(f"BigQuery num_rows: {table.num_rows:,}")
print(f"BigQuery num_cols: {len(table.schema)}")
print("=== SYNC RESULT: OK ===")

## 6) Verify

In [ ]:
sql = f"""
SELECT COUNT(*) AS n
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
"""
count_df = client.query(sql, location=LOCATION).to_dataframe()
print(count_df)

# Optional: week breakdown if WB column exists
cols = [f.name for f in client.get_table(table_id).schema]
wb_col = next((c for c in cols if c.lower() in ("wb", "week_beginning")), None)
if wb_col:
    sql_wb = f"""
    SELECT `{wb_col}` AS wb, COUNT(*) AS n
    FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
    GROUP BY 1
    ORDER BY 1 DESC
    LIMIT 15
    """
    print(client.query(sql_wb, location=LOCATION).to_dataframe())
else:
    print("No WB/week_beginning column found; skip week breakdown.")

print("Next: open dashboard → Refresh BigQuery")

## Optional: schedule later

Colab does **not** run forever on a timer by itself. Options:

1. **Re-run this notebook** when you need a refresh (manual)
2. **Vertex / Cloud Run + Cloud Scheduler** with a service account that has Sheet access
3. Keep a light **Apps Script** hourly trigger that only *calls* a Cloud Function

For most teams: run Colab **hourly/daily when needed**, then refresh the dashboard.